In [ ]:
# ==============================================================================
# LUNG CANCER CDSS: CLINICAL RADIOLOGIST SEGMENTATION PIPELINE (LIDC-IDRI)
# ==============================================================================
import os
import cv2
import glob
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support
from google.colab import files

print("Installing dependencies...")
!pip install -q --upgrade kaggle opencv-python scikit-learn mealpy

# 1. Authenticate Kaggle safely
os.environ['KAGGLE_API_TOKEN'] = "KGAT_4f540f54bf67bfec8ed2b84334125986"
!mkdir -p ~/.kaggle
!echo "KGAT_4f540f54bf67bfec8ed2b84334125986" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

# 2. Download the Gold Standard Radiologist-Labeled Dataset
print("\nDownloading the Clinical Lung-RADS Segmentation Dataset...")
!kaggle datasets download -d aminasirinechieb/lung-cancer-segmentation-lung-rads --force
!unzip -o -q lung-cancer-segmentation-lung-rads.zip -d lung_data/

# 3. Load the highly optimized .pkl clinical data
print("\nLoading Radiologist-Annotated Scans and Masks into Memory...")
train_pkl = glob.glob('lung_data/**/lung_cancer_train.pkl', recursive=True)[0]
df_train = pd.read_pickle(train_pkl)

IMG_SIZE = (128, 128)
X_raw, Y_raw = [], []

# Processing the first 1000 clinical slices
for idx, row in df_train.head(1000).iterrows():
    img = cv2.resize(row['hu_array'], IMG_SIZE)
    img = np.stack((img,)*3, axis=-1) 
    
    mask = cv2.resize(row['mask'].astype(np.float32), IMG_SIZE)
    mask = np.expand_dims(mask, axis=-1)
    
    X_raw.append(img)
    Y_raw.append(mask)

X = np.array(X_raw, dtype=np.float32)
Y = np.array(Y_raw, dtype=np.float32)
Y = np.where(Y > 0.5, 1.0, 0.0)

print(f"Data Loaded Successfully! Total Clinical Slices: {X.shape[0]}")
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

# ==========================================
# 4. MEDICAL U-NET ARCHITECTURE
# ==========================================
def dice_coef(y_true, y_pred, smooth=1):
    intersection = K.sum(K.flatten(y_true) * K.flatten(y_pred))
    return (2. * intersection + smooth) / (K.sum(K.flatten(y_true)) + K.sum(K.flatten(y_true)) + smooth)

def build_unet(input_shape=(128, 128, 3), dropout_rate=0.2):
    inputs = Input(input_shape)
    
    c1 = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    p1 = MaxPooling2D((2, 2))(c1)
    
    c2 = Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    p2 = MaxPooling2D((2, 2))(c2)
    
    c3 = Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
    c3 = Dropout(dropout_rate)(c3)
    
    u4 = UpSampling2D((2, 2))(c3)
    u4 = concatenate([u4, c2])
    c4 = Conv2D(64, (3, 3), activation='relu', padding='same')(u4)
    
    u5 = UpSampling2D((2, 2))(c4)
    u5 = concatenate([u5, c1])
    c5 = Conv2D(32, (3, 3), activation='relu', padding='same')(u5)
    
    outputs = Conv2D(1, (1, 1), activation='sigmoid')(c5)
    return Model(inputs=[inputs], outputs=[outputs])

# ==========================================
# 5. BASELINE MODEL (STANDARD ADAM)
# ==========================================
print("\n--- Training Baseline Adam U-Net ---")
model_adam = build_unet(dropout_rate=0.2)
model_adam.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=[dice_coef, 'accuracy'])
model_adam.fit(X_train, Y_train, validation_data=(X_val, Y_val), batch_size=16, epochs=10, verbose=1)
model_adam.save('unet_baseline_adam_real.h5')

# ==========================================
# 6. METAHEURISTIC OPTIMIZATION (GWO)
# ==========================================
print("\n--- Unleashing Grey Wolf Optimizer ---")
from mealpy.swarm_based import GWO
from mealpy import FloatVar

def fitness_function(solution):
    lr = float(solution[0])
    dropout = float(solution[1])
    
    trial_model = build_unet(dropout_rate=dropout)
    # FIX: Added 'accuracy' to match the evaluation output length!
    trial_model.compile(optimizer=Adam(learning_rate=lr), loss='binary_crossentropy', metrics=[dice_coef, 'accuracy'])
    trial_model.fit(X_train, Y_train, epochs=2, batch_size=32, verbose=0)
    
    val_loss, val_dice, val_acc = trial_model.evaluate(X_val, Y_val, verbose=0)
    return float(val_loss)

problem_dict = {
    "bounds": FloatVar(lb=[0.0001, 0.1], ub=[0.005, 0.6], name="hyperparams"),
    "obj_func": fitness_function,
    "minmax": "min"
}

model_gwo = GWO.OriginalGWO(epoch=4, pop_size=8)
model_gwo.solve(problem_dict)

best_lr = float(model_gwo.g_best.solution[0])
best_dropout = float(model_gwo.g_best.solution[1])
print(f"\n🏆 GWO Optimal Parameters Found -> Learning Rate: {best_lr:.5f}, Dropout: {best_dropout:.2f}")

# ==========================================
# 7. FINAL GWO MODEL TRAINING
# ==========================================
print("\n--- Training Final GWO Optimized U-Net ---")
model_optimized = build_unet(dropout_rate=best_dropout)
model_optimized.compile(optimizer=Adam(learning_rate=best_lr), loss='binary_crossentropy', metrics=[dice_coef, 'accuracy'])
model_optimized.fit(X_train, Y_train, validation_data=(X_val, Y_val), batch_size=16, epochs=10, verbose=1)
model_optimized.save('unet_optimized_gwo_real.h5')

# ==========================================
# 8. DOWNLOAD ARTIFACTS
# ==========================================
print("\nDownloading production models to your computer...")
files.download('unet_baseline_adam_real.h5')
files.download('unet_optimized_gwo_real.h5')